# Day 27 — Confusion Matrix and Threshold Tuning

**Level 2 · Day 27 · Data Science Track**

Analyze classification predictions using a confusion matrix and investigate how changing the classification threshold affects precision and recall.

## Objective
Understand the relationship between probability thresholds and classification performance.

## Tools
Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn · Jupyter Notebook

## Deliverables
- Confusion matrix visualization
- Results for multiple probability thresholds
- Precision-recall comparison
- Recommended threshold with justification


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


## 2. Load Dataset

We use the Breast Cancer Wisconsin Dataset. In scikit-learn, `0 = malignant` and `1 = benign`.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
print("Shape:", X.shape)
print(y.value_counts().sort_index())


## 3. Train Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train_scaled, y_train)
print("Model trained successfully.")


## 4. Predicted Probabilities

`predict_proba()` gives the estimated probability for each class. We use the probability of class 1.

In [ ]:
probabilities = model.predict_proba(X_test_scaled)[:, 1]
print(probabilities[:10])


## 5. Default Threshold — 0.50

In [ ]:
threshold = 0.50
y_pred = (probabilities >= threshold).astype(int)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


## 6. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.title("Confusion Matrix — Threshold 0.50")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 7. Test Multiple Thresholds

In [ ]:
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
results = []
for t in thresholds:
    pred = (probabilities >= t).astype(int)
    results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0)
    })
threshold_results = pd.DataFrame(results)
threshold_results.round(4)


## 8. Precision-Recall Comparison

In [ ]:
plt.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="Precision")
plt.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="Recall")
plt.plot(threshold_results["threshold"], threshold_results["f1"], marker="o", label="F1")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall and F1 vs Threshold")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 9. Recommended Threshold

In [ ]:
best = threshold_results.loc[threshold_results["f1"].idxmax()]
print("Recommended threshold:", best["threshold"])
print("Precision:", round(best["precision"], 4))
print("Recall:", round(best["recall"], 4))
print("F1:", round(best["f1"], 4))


## 10. Threshold Selection and Error Costs

- Lowering the threshold generally increases recall and may reduce precision.
- Raising the threshold generally increases precision and may reduce recall.
- If false negatives are costly, a lower threshold may be preferred.
- If false positives are costly, a higher threshold may be preferred.
- For this exercise, the recommended threshold is the tested threshold with the highest F1-score.


## 11. Interview Questions

**What is a confusion matrix?** A table showing true positives, true negatives, false positives and false negatives.

**How does changing the threshold affect recall?** Lowering the threshold usually makes more positive predictions, which can increase recall.

**Why might 0.5 not be optimal?** The best threshold depends on the application's precision-recall requirements and the cost of false positives and false negatives.


## Conclusion

Threshold tuning helps us choose a classification decision rule based on the desired balance between precision, recall, F1-score and real-world error costs.